# Fine-tune splits: historical train + May (70 / 15 / 15)

Build `train.csv`, `val.csv`, `test.csv` for **AHU 2-9 Blower DE A** fine-tuning:

| Split | Contents |
|-------|----------|
| **train** | **All** rows from `AHU_2_9_Blower_DE_A_30_min.csv` (historical 2023→~2025) **+** first **70%** of May (`AHU_2_9_Blower_DE_A_may_30_mins.csv`) by time |
| **val** | Next **15%** of May |
| **test** | Last **15%** of May |

Outputs: `finetune/data_AHU_2_9_Blower_DE_A/splits/`

Run all cells top to bottom.

In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import numpy as np
import pandas as pd

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "finetune").is_dir() and (REPO_ROOT.parent / "finetune").is_dir():
    REPO_ROOT = REPO_ROOT.parent

DATA_DIR = REPO_ROOT / "finetune" / "data_AHU_2_9_Blower_DE_A"
OUT_DIR = DATA_DIR / "splits"

HISTORICAL_CSV = DATA_DIR / "AHU_2_9_Blower_DE_A_30_min.csv"
MAY_CSV = DATA_DIR / "AHU_2_9_Blower_DE_A_may_30_mins.csv"

TIME_COL = "TIMESTAMP"
VALUE_COL = "Acceleration RMS"
SENSOR_COL = "SENSOR_DESC"
SENSOR_NAME = "AHU 2-9 Blower DE A"

# Historical file: use every row (val/test come from May only)
USE_FULL_HISTORICAL_IN_TRAIN = True

# May file only: 70% train / 15% val / 15% test (by time)
MAY_TRAIN_RATIO = 0.70
MAY_VAL_RATIO = 0.15
MAY_TEST_RATIO = 0.15

OUT_DIR.mkdir(parents=True, exist_ok=True)
print("REPO_ROOT:", REPO_ROOT)
print("HISTORICAL:", HISTORICAL_CSV)
print("MAY:", MAY_CSV)
print("OUT:", OUT_DIR)

In [ ]:
def parse_timestamp_series(series: pd.Series, *, strict: bool = True) -> pd.Series:
    raw = series.astype(str).str.strip()
    parsed = pd.to_datetime(raw, dayfirst=True, format="mixed", errors="coerce")
    for fmt in ("%Y-%m-%d %H:%M:%S", "%Y-%m-%d %H:%M"):
        mask = parsed.isna()
        if not mask.any():
            break
        parsed.loc[mask] = pd.to_datetime(raw.loc[mask], format=fmt, errors="coerce")
    if strict and int(parsed.isna().sum()):
        raise ValueError(f"Failed to parse {int(parsed.isna().sum())} timestamps.")
    return parsed


def load_and_prepare(path: Path, *, keep_columns: list[str] | None = None) -> pd.DataFrame:
    df = pd.read_csv(path, low_memory=False)
    if TIME_COL not in df.columns:
        raise ValueError(f"{path.name}: missing {TIME_COL}")
    if VALUE_COL not in df.columns:
        raise ValueError(f"{path.name}: missing {VALUE_COL}")
    df = df.copy()
    df[TIME_COL] = parse_timestamp_series(df[TIME_COL])
    if SENSOR_COL in df.columns:
        df[SENSOR_COL] = df[SENSOR_COL].astype(str).str.strip()
        df = df[df[SENSOR_COL] == SENSOR_NAME]
    df = df.sort_values(TIME_COL, kind="mergesort").reset_index(drop=True)
    df = df.drop_duplicates(subset=[TIME_COL], keep="last")
    if keep_columns is not None:
        missing = [c for c in keep_columns if c not in df.columns]
        if missing:
            raise ValueError(f"{path.name}: missing columns {missing}")
        df = df[keep_columns].copy()
    return df


def chrono_row_split(
    df: pd.DataFrame,
    train_ratio: float,
    val_ratio: float,
    test_ratio: float,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    rsum = float(train_ratio + val_ratio + test_ratio)
    if abs(rsum - 1.0) > 1e-6:
        raise ValueError(f"Ratios must sum to 1, got {rsum}")
    n = len(df)
    n_tr = int(np.floor(n * train_ratio))
    n_va = int(np.floor(n * val_ratio))
    n_te = n - n_tr - n_va
    if min(n_tr, n_va, n_te) < 1:
        raise ValueError(f"Empty split: n={n}, train={n_tr}, val={n_va}, test={n_te}")
    train_df = df.iloc[:n_tr].copy()
    val_df = df.iloc[n_tr : n_tr + n_va].copy()
    test_df = df.iloc[n_tr + n_va :].copy()
    return train_df, val_df, test_df


def summarize_frame(name: str, df: pd.DataFrame) -> dict:
    ts = df[TIME_COL]
    rms = pd.to_numeric(df[VALUE_COL], errors="coerce")
    return {
        "name": name,
        "rows": int(len(df)),
        "t_min": str(ts.min()),
        "t_max": str(ts.max()),
        "rms_mean": float(rms.mean()),
        "rms_median": float(rms.median()),
    }

In [ ]:
df_hist_full = load_and_prepare(HISTORICAL_CSV)
print("Historical rows:", len(df_hist_full))

CORE_COLS = [TIME_COL, SENSOR_COL, VALUE_COL]
df_may = load_and_prepare(MAY_CSV, keep_columns=CORE_COLS)
print("May rows:", len(df_may))

pd.DataFrame([summarize_frame("historical", df_hist_full), summarize_frame("may", df_may)])

In [ ]:
df_hist_train = df_hist_full.copy() if USE_FULL_HISTORICAL_IN_TRAIN else df_hist_full.iloc[: int(np.floor(len(df_hist_full) * 0.6))].copy()

print(f"Historical train pool: {len(df_hist_train):,} / {len(df_hist_full):,} rows (full historical)")
print(f"  {df_hist_train[TIME_COL].min()} -> {df_hist_train[TIME_COL].max()}")

df_may_train, df_may_val, df_may_test = chrono_row_split(
    df_may, MAY_TRAIN_RATIO, MAY_VAL_RATIO, MAY_TEST_RATIO
)

for label, part in [("may_train", df_may_train), ("may_val", df_may_val), ("may_test", df_may_test)]:
    s = summarize_frame(label, part)
    print(f"{label}: {s['rows']:,} rows  {s['t_min']} -> {s['t_max']}  RMS median {s['rms_median']:.3f}")

In [ ]:
hist_cols = list(df_hist_full.columns)

df_train = pd.concat(
    [df_hist_train, df_may_train.reindex(columns=hist_cols)],
    ignore_index=True,
)
df_train = df_train.sort_values(TIME_COL, kind="mergesort").reset_index(drop=True)

df_val = df_may_val.reindex(columns=hist_cols).sort_values(TIME_COL).reset_index(drop=True)
df_test = df_may_test.reindex(columns=hist_cols).sort_values(TIME_COL).reset_index(drop=True)

print("=== Final splits ===")
for name, part in [("train", df_train), ("val", df_val), ("test", df_test)]:
    s = summarize_frame(name, part)
    print(f"{name}: {s['rows']:,} rows | {s['t_min']} -> {s['t_max']} | RMS median {s['rms_median']:.3f}")

gap = df_may[TIME_COL].min() - df_hist_train[TIME_COL].max()
print(f"\nGap historical train end -> May start: {gap}")

In [ ]:
train_path = OUT_DIR / "train.csv"
val_path = OUT_DIR / "val.csv"
test_path = OUT_DIR / "test.csv"

df_train.to_csv(train_path, index=False)
df_val.to_csv(val_path, index=False)
df_test.to_csv(test_path, index=False)

manifest = {
    "sensor": SENSOR_NAME,
    "historical_csv": str(HISTORICAL_CSV),
    "may_csv": str(MAY_CSV),
    "use_full_historical_in_train": USE_FULL_HISTORICAL_IN_TRAIN,
    "may_split": {
        "train": MAY_TRAIN_RATIO,
        "val": MAY_VAL_RATIO,
        "test": MAY_TEST_RATIO,
    },
    "outputs": {
        "train": str(train_path),
        "val": str(val_path),
        "test": str(test_path),
    },
    "summary": {
        "train": summarize_frame("train", df_train),
        "val": summarize_frame("val", df_val),
        "test": summarize_frame("test", df_test),
        "hist_train_pool": summarize_frame("hist_train_pool", df_hist_train),
    },
}

manifest_path = OUT_DIR / "split_manifest.json"
with open(manifest_path, "w", encoding="utf-8") as fp:
    json.dump(manifest, fp, indent=2)

print("Wrote:")
print(" ", train_path)
print(" ", val_path)
print(" ", test_path)
print(" ", manifest_path)